In [1]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import json
import base64
from typing import Dict
import os
from anthropic import Anthropic

google_api_key = os.getenv("GOOGLE_API_KEY")
client_google = genai.Client(api_key = google_api_key)
model_google = "gemini-2.5-flash"

claude_api_key = os.getenv("ANTHROPIC_API_KEY")
client_claude = Anthropic(api_key = claude_api_key)
model_claude = "claude-sonnet-4-5-20250929"

csv_path = "quine_manual.csv"

In [2]:
def load_data(filepath: str):
    df = pd.read_csv(filepath, sep=";")
    data = []
    
    for idx, row in df.iterrows():
        word_text = f"\nNative Word {idx + 1}: {row['native_words']}\n"
        data.append({"type": "text", "content": word_text})
        
        for i in range(1, 6):
            img_path = row[f'image_example {i}']
            with open(img_path, 'rb') as f:
                img_b64 = base64.b64encode(f.read()).decode('utf-8')
                data.append({
                    "type": "image",
                    "mime_type": "image/png",
                    "data": img_b64
                })
    return data

In [3]:
def radical_translation_visual(prompt_template, model, n_iterations, csv_path):
    all_translations = {}
    
    items = load_data(csv_path)
    
    for i in range(n_iterations):
        try:
            if model == model_google: 
                content_payload = [prompt_template] 
                
                for item in items:
                    if item["type"] == "text":
                        content_payload.append(item["content"])
                    else:
                        content_payload.append({
                            "inline_data": {
                                "mime_type": item["mime_type"],
                                "data": item["data"]
                            }
                        })
                
                response = client_google.models.generate_content(
                    model=model,
                    contents=content_payload
                )
                translation = response.text

            elif model == model_claude:
                content_blocks = [{"type": "text", "text": prompt_template}]
                
                for item in items:
                    if item["type"] == "text":
                        content_blocks.append({"type": "text", "text": item["content"]})
                    else:
                        content_blocks.append({
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": item["mime_type"],
                                "data": item["data"]
                            }
                        })

                response = client_claude.messages.create(
                    model=model,
                    max_tokens=4096,
                    messages=[{"role": "user", "content": content_blocks}]
                )
                translation = response.content[0].text
            
            all_translations[f"translation_{i+1}"] = translation

        except Exception as e:
            print(f"Error: {e}")
            all_translations[f"translation_{i+1}"] = f"Error: {str(e)}"
            
    return all_translations

In [4]:
def full_experiment():
    QUINE_PROMPT = """You are a linguist that has to find proper 
    translations for what you see in the images in the manual. Think of
    the images as scenes to which the native pointed as he exclaimed
    the native word that refers to the scene.

    Respond with translations for each word from the manual:

    {manual}"""
    
    translations = radical_translation_visual(
        QUINE_PROMPT, 
        model_google, 
        3,  
        csv_path
    )

    for key, value in translations.items():
        print(f"--- {key} ---")
        print(value)
        print("\n")
  
full_experiment()

--- translation_1 ---
Here are the translations based on the provided images:

Native Word 1: $$%%
Translation: Door

Native Word 2: %&#%
Translation: Book

Native Word 3: $**$+
Translation: Deer


--- translation_2 ---
Here are the translations based on the images provided:

Native Word 1: $$%%
Translation: Door

Native Word 2: %&#%
Translation: Book

Native Word 3: $**$+
Translation: Deer


--- translation_3 ---
Here are the translations for the native words:

**Native Word 1: $$%%**
**Translation: Door** (This word encompasses various types of entrances and exits, from simple interior doors to heavy vault doors, and can also refer to door components like handles and hinges.)

**Native Word 2: %&#%**
**Translation: Book** (This refers to a bound volume of pages, whether closed, open, being read, or containing illustrations.)

**Native Word 3: $**$+**
**Translation: Deer** (This term refers to the animal, including specific features like its fur, face, antlers, and hooves, in various 

In [5]:
def full_experiment_():
    QUINE_PROMPT = """You are a linguist that has to find proper 
    translations for what you see in the images in the manual. Think of
    the images as scenes to which the native pointed as he exclaimed
    the native word that refers to the scene.

    Respond with translations for each word from the manual:

    {manual}"""
    
    translations = radical_translation_visual(
        QUINE_PROMPT, 
        model_claude, 
        3,  
        csv_path
    )

    for key, value in translations.items():
        print(f"--- {key} ---")
        print(value)
        print("\n")
  
full_experiment()

--- translation_1 ---
As a linguist examining the provided manual, here are the translations for each native word based on the scenes depicted:

Native Word 1: $$%%
Translation: **Door** (The images consistently show various types of doors: a standard interior door, a vault door, a glass office door, and a close-up of a doorknob, which is part of a door.)

Native Word 2: %&#%
Translation: **Book** (All images clearly depict books, whether closed, open, being read, or showing pages with text or drawings.)

Native Word 3: $**$+
Translation: **Deer** (The collection of images shows a deer in various poses and close-ups, highlighting its characteristics like antlers, fur, and hooves.)


--- translation_2 ---
As a linguist examining these scenes and their associated native words, here are my proposed translations:

Native Word 1: $$%%
Translation: **Door** (or **Portal**, encompassing entrances and exits)

Native Word 2: %&#%
Translation: **Book** (or **Volume**, **Text**)

Native Word 3: $